# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 ("The Anatomy of Growing Content"):**

**Finding:** Growing pages average 3,180 words vs 2,311 for declining pages, and are younger (184d vs 230d), based on the paper's "up"/"down" trend labels.

**My methodology question**: The paper defines trend direction from a 30-day-vs-previous-30-day impression change (up: >10%, down: >10%, flat: insufficient data). For pages with low absolute impression counts, a 10% swing can easily be noise rather than a real trend. I ran into exactly this with my own decline label in w04/w05, where a fixed percentage threshold on small-volume pages made the classification threshold-sensitive rather than meaningful. Is there a minimum-impression floor applied before assigning "up"/"down," or could low-traffic pages be more likely to land in either bucket by chance, which would inflate the reported word-count and age gap between the two groups? I'd want to see the direction label re-run with a volume floor (e.g., 100+ impressions/month) before fully trusting the size of that gap.

**Finding 4 ("The Freshness Multiplier"):**

**Finding:** Growth-to-decline ratio by freshness window 31-90 days: 7.88:1, 181-360 days: 3.13:1, 361+ days: 283:1.

**My methodology question**: The paper itself flags the 361+ bucket as unstable (283 growing vs just 1 declining page) which is exactly the right instinct, and the kind of honesty I had to apply to my own Signal 1 check in w04, where an n=18 bucket showing a striking decline rate still only earned a MIXED verdict, not CONFIRMED, because the sample was too small to trust. My question: is that same n-based caution applied consistently to the other buckets in this table, or only to the one that happened to look extreme? The 181-360 bucket (3.13:1) reads as solid, but I'd want the actual n behind each ratio shown alongside it the same way I now know to ask of my own numbers.

In [22]:
import pandas as pd
findings = pd.DataFrame([
    {"finding": "#1 Anatomy of Growing Content", "claim": "Growing pages: 3,180 words / 184d age; Declining: 2,311 words / 230d age", "my_question": "Is there a volume floor on the up/down label?"},
    {"finding": "#4 Freshness Multiplier", "claim": "Growth:decline ratio by freshness window — 31-90d: 7.88:1, 181-360d: 3.13:1, 361+: 283:1 (n=1 declining)", "my_question": "Is n reported/thresholded consistently across all buckets, not just the flagged one?"},
])
findings

,finding,claim,my_question
0,#1 Anatomy of Growing Content,"Growing pages: 3,180 words / 184d age; Declini...",Is there a volume floor on the up/down label?
1,#4 Freshness Multiplier,Growth:decline ratio by freshness window — 31-...,Is n reported/thresholded consistently across ...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [21]:
%pip -q install duckdb scikit-learn
import os, getpass
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
FEATURE_MONTH, OUTCOME_MONTH = '2026-03', '2026-04'

monthly = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN month = '{FEATURE_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_march,
        SUM(CASE WHEN month = '{FEATURE_MONTH}' THEN gsc_clicks ELSE 0 END) AS clk_march,
        AVG(CASE WHEN month = '{FEATURE_MONTH}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_march,
        COUNT(*) FILTER (WHERE month = '{FEATURE_MONTH}' AND gsc_impressions > 0) AS active_days_march,
        SUM(CASE WHEN month = '{OUTCOME_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_april
    FROM {FACT}
    WHERE month IN ('{FEATURE_MONTH}', '{OUTCOME_MONTH}')
    GROUP BY 1,2 HAVING imp_march >= 100
""").df()
monthly['ctr_march'] = (monthly['clk_march'] / monthly['imp_march']).round(4)
monthly['declined_next_month'] = (monthly['imp_april'] < 0.8 * monthly['imp_march']).astype(int)

content = con.sql(f"""
    SELECT content_hash_id, DATE_DIFF('day', content_updated_date, DATE '{FEATURE_MONTH}-01') AS days_since_last_update
    FROM {DIM_CONTENT}
""").df()
df = monthly.merge(content, on='content_hash_id', how='left')
df.loc[df['days_since_last_update'] < 0, 'days_since_last_update'] = pd.NA
df['staleness_unknown'] = df['days_since_last_update'].isna().astype(int)
df['days_since_last_update_filled'] = df['days_since_last_update'].fillna(-1)

FEATURES = ['imp_march', 'ctr_march', 'pos_march', 'active_days_march',
            'days_since_last_update_filled', 'staleness_unknown']
model_df = df.dropna(subset=['pos_march']).copy()
X, y, groups = model_df[FEATURES], model_df['declined_next_month'], model_df['client_hash_id']

def run_rf(X_train, X_test, y_train, y_test, label):
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_train, y_train)
    scores = rf.predict_proba(X_test)[:, 1]
    return {'split': label, 'AUC': round(roc_auc_score(y_test, scores), 3), 'base_rate': round(y_test.mean(), 3)}, rf

# BEFORE: random row split — the dishonest one
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
random_result, random_rf = run_rf(Xr_train, Xr_test, yr_train, yr_test, 'Random split (before)')

# AFTER: grouped-by-client split — the honest one (same as w05)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
grouped_result, grouped_rf = run_rf(X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx], 'Grouped by client (after)')

comparison = pd.DataFrame([random_result, grouped_result])
comparison

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,AUC,base_rate
0,Random split (before),0.677,0.517
1,Grouped by client (after),0.673,0.484


The gap between the random-split AUC (0.677) and the grouped-split AUC (0.673) is essentially zero (0.004). Unlike the LR scaling issue I found in w05, this result doesn't reveal a hidden memorization effect switching from a random split to a client-grouped split barely moved Random Forest's score at all here. That's a real, honest finding worth naming plainly rather than over-explaining: for this particular feature set (page-level averages with no client-identifying values), client identity doesn't appear to be doing much hidden work. The base rate differs more (0.517 vs 0.484) simply because grouping by client can't guarantee the same class balance a stratified random split can. I wouldn't extend this to claim grouped splits are unnecessary in general only that for this specific model, the two splits happened to agree closely.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
print("LEAKAGE AUDIT — final feature set:", FEATURES)
print()
print("[✓] Timeline: all 6 features built from March only; label (declined_next_month) from April only.")
print("[✓] No label-derived columns: imp_april/clk_april never appear in FEATURES.")
print("[✓] No product flags used: health_score, priority_score, action_type, is_quick_win absent.")
print("[✓] Split grouped by client_hash_id (Section 2).")
print(f"[✓] Base rate printed: {y.mean():.3f} declined overall.")
print()

# Deliberate-leak verification: does the test harness even notice a real leak?
X_leak = X.copy()
X_leak['imp_april_LEAK'] = model_df['imp_april']  # straight from the label's own formula

Xl_train, Xl_test, yl_train, yl_test = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
rf_leak = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(Xl_train, yl_train)
leak_auc = roc_auc_score(yl_test, rf_leak.predict_proba(Xl_test)[:, 1])

print(f"Honest AUC (no leak): ~{grouped_result['AUC']}")
print(f"Leaky AUC (imp_april added): {leak_auc:.3f}")
print("Harness confirmed working:", "YES — score jumped toward 1.0" if leak_auc > 0.9 else "NO — investigate the harness")

# remove it — keep only the honest features
del X_leak

LEAKAGE AUDIT — final feature set: ['imp_march', 'ctr_march', 'pos_march', 'active_days_march', 'days_since_last_update_filled', 'staleness_unknown']

[✓] Timeline: all 6 features built from March only; label (declined_next_month) from April only.
[✓] No label-derived columns: imp_april/clk_april never appear in FEATURES.
[✓] No product flags used: health_score, priority_score, action_type, is_quick_win absent.
[✓] Split grouped by client_hash_id (Section 2).
[✓] Base rate printed: 0.517 declined overall.

Honest AUC (no leak): ~0.673
Leaky AUC (imp_april added): 0.914
Harness confirmed working: YES — score jumped toward 1.0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [15]:
importances = pd.Series(grouped_rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances

,0
ctr_march,0.372151
active_days_march,0.328846
imp_march,0.125732
pos_march,0.094928
days_since_last_update_filled,0.044702
staleness_unknown,0.033641


In [17]:
top1, top2 = importances.index[0], importances.index[1]
val1, val2 = importances.iloc[0], importances.iloc[1]
print(top1, val1, top2, val2)

ctr_march 0.3721513196560473 active_days_march 0.3288460215160088


In [20]:
importances = pd.Series(grouped_rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
top1, top2 = importances.index[0], importances.index[1]
val1, val2 = importances.iloc[0], importances.iloc[1]

print("ORIGINAL (from w05):")
print("a page's own click-through behavior and how consistently it showed impressions "
      "across March are reasonable early signals of a coming drop")
print()
print("REWRITTEN (safe language):")
print(f"In this dataset and time window, {top1} and {top2} had the highest observed "
      f"importance scores ({val1:.3f} and {val2:.3f}) in the Random Forest model. "
      f"This is a measured, directional pattern from one month of data and one client "
      f"sample — a decision-support signal, not a general claim that these features "
      f"predict decline everywhere.")

ORIGINAL (from w05):
a page's own click-through behavior and how consistently it showed impressions across March are reasonable early signals of a coming drop

REWRITTEN (safe language):
In this dataset and time window, ctr_march and active_days_march had the highest observed importance scores (0.372 and 0.329) in the Random Forest model. This is a measured, directional pattern from one month of data and one client sample — a decision-support signal, not a general claim that these features predict decline everywhere.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.